<a href="https://colab.research.google.com/github/Shalom-u5841413/Shalomu5841413-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** 23:00  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

i'm going to look at the different tables first to understand what information is available and how the data is organised. i won't make any changes yet as i want to identify what needs checking before cleaning.Add text here

In [5]:
from IPython.display import display

# Loop through each table to print shape and show the first 5 rows
for name, df in tables.items():
    print(f"=== Table: {name} ({df.shape[0]:,} rows, {df.shape[1]} columns) ===")
    display(df.head(5))
    print("\n" + "="*50 + "\n")

=== Table: orders (14,494 rows, 5 columns) ===


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00




=== Table: order_status_events (56,812 rows, 4 columns) ===


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z
3,E021428,O005468,delivered,2026-04-15T13:43:36Z
4,E037134,O009483,picked_up,2026-05-19T12:18:18Z




=== Table: order_lines (36,120 rows, 5 columns) ===


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5
3,O000002,3,Burrito,3,8.0
4,O000003,1,Fries,1,3.0


... (2 outputs left out)



=== Table: restaurants (67 rows, 5 columns) ===


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01




=== Table: customers (4,200 rows, 3 columns) ===


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds
3,C00004,2024-09-14,London
4,C00005,2024-01-25,Manchester


the data contains 6 tables with information on orders, status updates, items purchased, payments, restaurants and customers. i've noticed that some tables have more rows than the orders table, which suggests that one order can have multiple records.

this is worth considering when combining the tables as it could lead to orders being counted more than once. i'll check the data for missing values and duplicates before making any changes.

now i want to check all 6 tables for missing values and duplicates to see if there are any issues that need cleaning. i'll also check whether the ID columns contain repeated values, as some tables might have multiple records for the same order.

**AI Prompt:** i've looked at the 6 tables and now want to check them for missing values and duplicate rows. i also want to check whether order_id is repeated in the orders, order_lines, payments and order_status_events tables, and whether restaurant_id and customer_id are repeated in their tables. can you give me simple python code to do this for all the tables? don't change anything yet.

In [6]:
# 1. Check for missing values and completely duplicate rows in each table
for name, df in tables.items():
    missing_count = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()
    print(f"Table '{name}':")
    print(f"  - Missing values: {missing_count}")
    print(f"  - Completely duplicate rows: {duplicate_rows}")
    print()

# 2. Check key columns for duplicates/repeats
print("=== Duplicate Key Checks ===")

# order_id checks
for tbl in ["orders", "order_lines", "payments", "order_status_events"]:
    if tbl in tables:
        df = tables[tbl]
        total_ids = len(df["order_id"])
        unique_ids = df["order_id"].nunique()
        print(f"'{tbl}' table 'order_id': {total_ids} total values, {unique_ids} unique values (Repeated: {total_ids > unique_ids})")

# restaurant_id check
if "restaurants" in tables:
    df = tables["restaurants"]
    total_res = len(df["restaurant_id"])
    unique_res = df["restaurant_id"].nunique()
    print(f"'restaurants' table 'restaurant_id': {total_res} total values, {unique_res} unique values (Repeated: {total_res > unique_res})")

# customer_id check
if "customers" in tables:
    df = tables["customers"]
    total_cust = len(df["customer_id"])
    unique_cust = df["customer_id"].nunique()
    print(f"'customers' table 'customer_id': {total_cust} total values, {unique_cust} unique values (Repeated: {total_cust > unique_cust})")

Table 'orders':
  - Missing values: 0
  - Completely duplicate rows: 0

Table 'order_status_events':
  - Missing values: 0
  - Completely duplicate rows: 0

Table 'order_lines':
  - Missing values: 0
  - Completely duplicate rows: 0

Table 'payments':
  - Missing values: 0
  - Completely duplicate rows: 30

Table 'restaurants':
  - Missing values: 0
  - Completely duplicate rows: 0

Table 'customers':
  - Missing values: 0
  - Completely duplicate rows: 0

=== Duplicate Key Checks ===
'orders' table 'order_id': 14494 total values, 14494 unique values (Repeated: False)
'order_lines' table 'order_id': 36120 total values, 14494 unique values (Repeated: True)
'payments' table 'order_id': 16423 total values, 14494 unique values (Repeated: True)
'order_status_events' table 'order_id': 56812 total values, 14494 unique values (Repeated: True)
'restaurants' table 'restaurant_id': 67 total values, 64 unique values (Repeated: True)
'customers' table 'customer_id': 4200 total values, 4200 unique v

the checks showed no missing values across the 6 tables, but i found 30 completely duplicated rows in payments. i also noticed that restaurants has 67 records but only 64 unique restaurant IDs.

the repeated order IDs in payments, order_lines and order_status_events aren't necessarily errors as one order can have multiple records. i'll investigate the payment duplicates and repeated restaurant IDs before making changes.

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

i want my final table to show one row for each combination of week, cuisine and city, with the number of orders, net revenue and average delivery time.

i'll start with the orders table, which has 14,494 rows, as each row represents one order. i'll then clean the other tables before combining them. i'll use order_status_events to identify delivered and cancelled orders and calculate delivery times, payments to calculate net revenue, and restaurants to add cuisine and city.

i've noticed that payments has 30 duplicate rows, so i'll remove the extra copies. restaurants also has repeated IDs, which i'll investigate using effective_from before joining, as these might represent different versions of the same restaurant.

i expect the order-level table to remain at 14,494 rows after combining the information, before grouping it into weekly results. i'll check the row counts after each join to make sure orders aren't accidentally duplicated.

for the calculations, i'll use the order placement date to decide the week, count successfully delivered orders, calculate net revenue as charges minus refunds, and measure delivery time from when an order was placed to when it was delivered. i'll check that the timestamps are valid before calculating the durations.

finally, i'll group the results by week, cuisine and city and compare the totals with the cleaned data to check that the reporting table is accurate.Add text here

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

i've found 30 completely duplicated rows in payments, which could cause the same payment to be counted more than once. i'm going to remove the extra copies in a new table while keeping the original unchanged.

i expect the number of records to decrease from 16,423 to 16,393.

**AI Prompt:** i've found 30 duplicate rows in payments. can you give me python code to remove the extra copies in a new payments_clean table without changing the original? also check the number of rows before and after and whether any duplicates remain.

In [7]:
# Access the original payments DataFrame
payments_orig = tables["payments"]

# Count rows before cleaning
rows_before = len(payments_orig)
print(f"Rows in payments before deduplication: {rows_before}")

# Remove completely duplicate rows and store in a new DataFrame
payments_clean = payments_orig.drop_duplicates()
tables["payments_clean"] = payments_clean

# Count rows after cleaning
rows_after = len(payments_clean)
remaining_duplicates = payments_clean.duplicated().sum()

print(f"Rows in payments_clean after deduplication: {rows_after} (Expected: {rows_before - 30})")
print(f"Completely duplicate rows remaining: {remaining_duplicates}")

Rows in payments before deduplication: 16423
Rows in payments_clean after deduplication: 16393 (Expected: 16393)
Completely duplicate rows remaining: 0


after removing the duplicates, the number of payment records decreased from 16,423 to 16,393, which matches what i expected. there are also no exact duplicates remaining, so this should prevent the same payment record from being counted twice in the revenue calculations.

i noticed earlier that the restaurants table has 67 rows but only 64 unique restaurant IDs. this could mean some restaurants have more than one record, possibly because their information changed over time.

i'm going to investigate the repeated IDs and their effective_from dates before deciding how to handle them. this is important because joining repeated restaurant IDs to orders could increase the number of rows.

**AI Prompt:** i've noticed that restaurants has 67 rows but only 64 unique restaurant IDs. i want to see which restaurants are repeated and compare their cuisine, city and effective_from dates to understand why. can you give me python code to display these records and check whether any restaurant has the same effective_from date more than once? don't remove or change anything yet.

In [8]:
import pandas as pd

restaurants_df = tables["restaurants"]

# 1. Identify which restaurant IDs are repeated
duplicated_ids = restaurants_df[restaurants_df.duplicated(subset=['restaurant_id'], keep=False)]['restaurant_id'].unique()
print(f"Repeated Restaurant IDs: {list(duplicated_ids)}\n")

# 2. Display the full records for these repeated restaurants sorted by ID and effective_from
print("=== Repeated Restaurant Records ===")
repeated_records = restaurants_df[restaurants_df['restaurant_id'].isin(duplicated_ids)].sort_values(by=['restaurant_id', 'effective_from'])
display(repeated_records)

# 3. Check if any restaurant has the same effective_from date more than once
dup_effective_dates = restaurants_df.duplicated(subset=['restaurant_id', 'effective_from']).sum()
print(f"\nNumber of records with duplicate [restaurant_id, effective_from] combinations: {dup_effective_dates}")

Repeated Restaurant IDs: ['R007', 'R021', 'R040']

=== Repeated Restaurant Records ===


,restaurant_id,restaurant_name,cuisine,city,effective_from
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
7,R007,Royal Japanese 7,Japanese,London,2026-04-14
51,R021,Grand Thai 21,Thai,Manchester,2025-01-01
6,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
48,R040,Corner Vegan 40,Vegan,London,2025-01-01
35,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27



Number of records with duplicate [restaurant_id, effective_from] combinations: 0


the results showed that 3 restaurants have two records each, with different effective_from dates and cities. there were also no repeated combinations of restaurant ID and effective date.

i've decided to keep these records as they appear to show changes in restaurant information over time. removing them could mean orders are assigned to the wrong city, so i'll need to match each order to the restaurant information that was valid when it was placed.

now i want to add the restaurant cuisine and city to each order. since 3 restaurants have changed cities, i need to use the effective_from dates so that orders are matched to the correct city at the time they were placed.

i expect the number of orders to stay at 14,494 after combining the tables, with no orders duplicated or lost.

**AI Prompt:** i've found that 3 restaurants have changed cities on different dates. i want to combine orders with restaurants using restaurant_id, but make sure each order gets the cuisine and city that applied when it was placed using order_ts and effective_from. can you give me python code to do this without changing the original tables? also check that the number of orders stays at 14494 and that no orders have missing restaurant information.

In [10]:
import pandas as pd

# Get the original DataFrames
orders_df = tables["orders"].copy()
restaurants_df = tables["restaurants"].copy()

# Convert timestamps to datetime and strip timezone info so they are compatible
orders_df["order_ts"] = pd.to_datetime(orders_df["order_ts"]).dt.tz_localize(None)
restaurants_df["effective_from"] = pd.to_datetime(restaurants_df["effective_from"]).dt.tz_localize(None)

# 1. Merge all restaurant records with orders on restaurant_id
merged = orders_df.merge(restaurants_df, on="restaurant_id", how="left")

# 2. Keep only the records where the order occurred on or after the restaurant's effective_from date
valid_records = merged[merged["order_ts"] >= merged["effective_from"]]

# 3. For each order, pick the restaurant record with the latest (maximum) effective_from date
orders_with_restaurants = (
    valid_records.sort_values(by="effective_from", ascending=False)
    .drop_duplicates(subset=["order_id"], keep="first")
    .sort_values(by="order_ts")
)

# Store the clean result in tables dict
tables["orders_with_restaurants"] = orders_with_restaurants

# --- Validation Checks ---
final_count = len(orders_with_restaurants)
missing_cuisine = orders_with_restaurants["cuisine"].isnull().sum()
missing_city = orders_with_restaurants["city"].isnull().sum()

print(f"Validation Results:")
print(f"- Number of orders: {final_count} (Expected: 14494)")
print(f"- Orders with missing cuisine: {missing_cuisine} (Expected: 0)")
print(f"- Orders with missing city: {missing_city} (Expected: 0)")

# Display first few rows of the matched result
display(orders_with_restaurants[["order_id", "restaurant_id", "order_ts", "cuisine", "city"]].head())

Validation Results:
- Number of orders: 14494 (Expected: 14494)
- Orders with missing cuisine: 0 (Expected: 0)
- Orders with missing city: 0 (Expected: 0)


,order_id,restaurant_id,order_ts,cuisine,city
0,O000001,R052,2026-03-01 09:13:23,Burgers,London
1,O000002,R027,2026-03-01 09:15:33,Chinese,London
2,O000003,R044,2026-03-01 09:23:42,Burgers,Birmingham
3,O000004,R049,2026-03-01 09:28:14,Italian,Birmingham
4,O000005,R009,2026-03-01 11:32:35,Italian,Birmingham


the code produced a TypeError because the order dates and restaurant effective dates have different timezone formats. this means python couldn't compare them to decide which restaurant record applied when each order was placed.

i'm going to ask gemini to correct the date formats without changing the original tables, then rerun the code to see whether all 14,494 orders are matched correctly.

**AI Prompt:** my code gave me a TypeError saying it can't compare tz-naive and tz-aware dates. i think order_ts and effective_from have different timezone formats. can you fix my existing code so both dates use UTC and can be compared? keep the rest of the code the same and check that i still have 14494 orders with no missing cuisine or city. please give me the full corrected code.

In [11]:
import pandas as pd

# Get copy of original DataFrames to keep them clean
orders_df = tables["orders"].copy()
restaurants_df = tables["restaurants"].copy()

# Convert both timestamp columns to UTC to resolve timezone mismatch
orders_df["order_ts"] = pd.to_datetime(orders_df["order_ts"]).dt.tz_convert("UTC")
restaurants_df["effective_from"] = pd.to_datetime(restaurants_df["effective_from"]).dt.tz_localize("UTC")

# 1. Merge all restaurant records with orders on restaurant_id
merged = orders_df.merge(restaurants_df, on="restaurant_id", how="left")

# 2. Keep only the records where the order occurred on or after the restaurant's effective_from date
valid_records = merged[merged["order_ts"] >= merged["effective_from"]]

# 3. For each order, pick the restaurant record with the latest (maximum) effective_from date
orders_with_restaurants = (
    valid_records.sort_values(by="effective_from", ascending=False)
    .drop_duplicates(subset=["order_id"], keep="first")
    .sort_values(by="order_ts")
)

# Store result in tables dict
tables["orders_with_restaurants"] = orders_with_restaurants

# --- Validation Checks ---
final_count = len(orders_with_restaurants)
missing_cuisine = orders_with_restaurants["cuisine"].isnull().sum()
missing_city = orders_with_restaurants["city"].isnull().sum()

print(f"Validation Results:")
print(f"- Number of orders: {final_count} (Expected: 14494)")
print(f"- Orders with missing cuisine: {missing_cuisine} (Expected: 0)")
print(f"- Orders with missing city: {missing_city} (Expected: 0)")

# Display the top rows
display(orders_with_restaurants[["order_id", "restaurant_id", "order_ts", "cuisine", "city"]].head())

Validation Results:
- Number of orders: 14494 (Expected: 14494)
- Orders with missing cuisine: 0 (Expected: 0)
- Orders with missing city: 0 (Expected: 0)


,order_id,restaurant_id,order_ts,cuisine,city
0,O000001,R052,2026-03-01 09:13:23+00:00,Burgers,London
1,O000002,R027,2026-03-01 09:15:33+00:00,Chinese,London
2,O000003,R044,2026-03-01 09:23:42+00:00,Burgers,Birmingham
3,O000004,R049,2026-03-01 09:28:14+00:00,Italian,Birmingham
4,O000005,R009,2026-03-01 11:32:35+00:00,Italian,Birmingham


after correcting the date formats, the code ran successfully and all 14,494 orders were retained, with no missing cuisine or city information. this matches what i expected and suggests the restaurant information has been joined correctly.

however, i've noticed that some cities have inconsistent capitalisation, such as London and london. i'll standardise these before creating the final reporting table so they aren't treated as different cities.

i'm going to check that no order IDs were repeated during the join and make the city names consistent. i expect to still have 14,494 orders and no repeated IDs.

**AI Prompt:** my orders_with_restaurants table has 14494 rows and no missing cuisine or city, but i've noticed some city names are written differently like London and london. can you give me simple python code to check for repeated order IDs and standardise the city names? keep the original tables unchanged and check the row count and unique cities afterwards.

In [12]:
# Check for repeated order IDs
order_id_duplicates = tables["orders_with_restaurants"]["order_id"].duplicated().sum()
print(f"Number of repeated order IDs: {order_id_duplicates}")

# Standardise city names to Title Case to fix inconsistent capitalisation (e.g. london, LEEDS)
tables["orders_with_restaurants"]["city"] = tables["orders_with_restaurants"]["city"].str.strip().str.title()

# Check the unique cities and row count afterwards
unique_cities = tables["orders_with_restaurants"]["city"].unique()
final_row_count = len(tables["orders_with_restaurants"])

print(f"Unique cities after standardisation: {list(unique_cities)}")
print(f"Final row count: {final_row_count} (Expected: 14494)")

Number of repeated order IDs: 0
Unique cities after standardisation: ['London', 'Birmingham', 'Leeds', 'Manchester', 'Bristol']
Final row count: 14494 (Expected: 14494)


the checks showed that there are still 14,494 orders with no repeated order IDs. the city names have also been standardised into 5 consistent categories, which means the same city won't be treated as two different groups because of capitalisation.

this matches what i expected, so i can now use this table for the weekly report.

now i want to use the order_status_events table to identify which orders were delivered and calculate how long delivery took.

since one order can have multiple status updates, i'll need to make sure each order only appears once in the new table. i'll calculate delivery time using the difference between when the order was placed and when it was delivered, and check for missing or negative durations.

**AI Prompt:** i want to use order_status_events to find which orders were delivered and calculate delivery time in minutes. can you give me python code to find the placed and delivered times for each order_id, then calculate the difference between them? i want one row per order and checks for repeated order IDs, missing times and negative durations. don't change the original table.

In [13]:
import pandas as pd
import numpy as np

# Get copy of original status events to keep it clean
events_df = tables["order_status_events"].copy()

# Standardise timestamps to datetime
events_df["event_ts"] = pd.to_datetime(events_df["event_ts"]).dt.tz_convert("UTC")

# 1. Filter and pivot to get 'placed' and 'delivered' times for each order
# We will extract 'placed' (often represented by the earliest status or 'placed' status) and 'delivered' statuses.
placed_events = events_df[events_df["status"] == "placed"].rename(columns={"event_ts": "placed_ts"})
delivered_events = events_df[events_df["status"] == "delivered"].rename(columns={"event_ts": "delivered_ts"})

# Merge these together on order_id
order_timings = pd.merge(
    placed_events[["order_id", "placed_ts"]],
    delivered_events[["order_id", "delivered_ts"]],
    on="order_id",
    how="outer"
)

# 2. Calculate delivery duration in minutes
order_timings["delivery_duration_mins"] = (
    order_timings["delivered_ts"] - order_timings["placed_ts"]
).dt.total_seconds() / 60.0

# Store result in tables dict
tables["order_delivery_performance"] = order_timings

# --- Validation Checks ---
print("=== Validation Results ===")
print(f"Total unique orders analyzed: {len(order_timings):,}")

# Check for repeated order IDs
duplicate_order_ids = order_timings["order_id"].duplicated().sum()
print(f"- Repeated order IDs: {duplicate_order_ids}")

# Check for missing times
missing_placed = order_timings["placed_ts"].isnull().sum()
missing_delivered = order_timings["delivered_ts"].isnull().sum()
print(f"- Orders missing 'placed' timestamp: {missing_placed}")
print(f"- Orders missing 'delivered' timestamp (e.g. cancelled/in-progress): {missing_delivered}")

# Check for negative durations
negative_durations = (order_timings["delivery_duration_mins"] < 0).sum()
print(f"- Orders with negative delivery durations: {negative_durations}")

# Display preview
display(order_timings.head())

=== Validation Results ===
Total unique orders analyzed: 14,634
- Repeated order IDs: 140
- Orders missing 'placed' timestamp: 0
- Orders missing 'delivered' timestamp (e.g. cancelled/in-progress): 1085
- Orders with negative delivery durations: 16


,order_id,placed_ts,delivered_ts,delivery_duration_mins
0,O000001,2026-03-01 09:13:23+00:00,2026-03-01 10:13:39+00:00,60.266667
1,O000002,2026-03-01 09:15:33+00:00,2026-03-01 10:01:53+00:00,46.333333
2,O000003,2026-03-01 09:23:42+00:00,2026-03-01 10:02:45+00:00,39.050000
3,O000004,2026-03-01 09:28:14+00:00,2026-03-01 10:10:41+00:00,42.450000
4,O000005,2026-03-01 11:32:35+00:00,NaT,NaN


the checks showed that the code produced 14,634 rows rather than the expected 14,494, with 140 repeated order IDs. this suggests that some orders have multiple status events which haven't been combined correctly.

i also found 1,085 rows without a delivered timestamp and 16 negative delivery durations. i won't use this table for the averages yet as the repeated orders and negative times could affect the results.

i'm going to correct the delivery-time table so each order only appears once. i'll also investigate the negative delivery durations and exclude them from the average calculations without removing the orders themselves.

i expect the corrected table to have 14,494 rows with no repeated order IDs. i'll keep missing or invalid delivery times as missing values so they don't affect the average.

**AI Prompt:** my delivery time code produced 14634 rows instead of 14494, with 140 repeated order IDs, 1085 missing delivered times and 16 negative durations. can you fix my existing code so there's only one row per order_id? use the earliest placed event and the earliest delivered event after placement, and keep orders without a valid delivery time but set their duration to missing. don't change the original data. check the row count, repeated IDs and missing durations afterwards.

In [14]:
import pandas as pd
import numpy as np

# Get a copy of the original order status events
events_df = tables["order_status_events"].copy()

# Convert timestamps to timezone-aware datetime in UTC
events_df["event_ts"] = pd.to_datetime(events_df["event_ts"]).dt.tz_convert("UTC")

# 1. Get the earliest 'placed' timestamp for each order_id
placed_df = (
    events_df[events_df["status"] == "placed"]
    .sort_values("event_ts")
    .drop_duplicates(subset=["order_id"], keep="first")[["order_id", "event_ts"]]
    .rename(columns={"event_ts": "placed_ts"})
)

# 2. Get the 'delivered' events
delivered_events = events_df[events_df["status"] == "delivered"][["order_id", "event_ts"]].rename(columns={"event_ts": "delivered_ts"})

# 3. Join them to evaluate which delivery happened after placement
merged_events = pd.merge(placed_df, delivered_events, on="order_id", how="left")

# Filter to only keep deliveries that happened strictly after the placement time
valid_deliveries = merged_events[(merged_events["delivered_ts"] > merged_events["placed_ts"]) | (merged_events["delivered_ts"].isnull())]

# For each order, pick the earliest valid delivery timestamp
earliest_delivery = (
    valid_deliveries.sort_values("delivered_ts")
    .drop_duplicates(subset=["order_id"], keep="first")
)

# 4. Perform a left join from the main orders table to ensure we preserve all 14,494 orders
all_orders = tables["orders"][["order_id"]].copy()
order_delivery_perf = pd.merge(all_orders, earliest_delivery, on="order_id", how="left")

# 5. Calculate delivery duration in minutes (will automatically be NaN if delivered_ts is missing)
order_delivery_perf["delivery_duration_mins"] = (
    order_delivery_perf["delivered_ts"] - order_delivery_perf["placed_ts"]
).dt.total_seconds() / 60.0

# Save the result in the tables dictionary
tables["order_delivery_performance"] = order_delivery_perf

# --- Validation Checks ---
final_row_count = len(order_delivery_perf)
duplicate_ids = order_delivery_perf["order_id"].duplicated().sum()
missing_durations = order_delivery_perf["delivery_duration_mins"].isnull().sum()
negative_durations = (order_delivery_perf["delivery_duration_mins"] < 0).sum()

print("=== Validation Results ===")
print(f"- Final row count: {final_row_count} (Expected: 14494)")
print(f"- Repeated order IDs: {duplicate_ids} (Expected: 0)")
print(f"- Missing delivery durations: {missing_durations}")
print(f"- Negative delivery durations: {negative_durations} (Expected: 0)")

# Display preview
display(order_delivery_perf.head())

=== Validation Results ===
- Final row count: 14494 (Expected: 14494)
- Repeated order IDs: 0 (Expected: 0)
- Missing delivery durations: 1101
- Negative delivery durations: 0 (Expected: 0)


,order_id,placed_ts,delivered_ts,delivery_duration_mins
0,O000001,2026-03-01 09:13:23+00:00,2026-03-01 10:13:39+00:00,60.266667
1,O000002,2026-03-01 09:15:33+00:00,2026-03-01 10:01:53+00:00,46.333333
2,O000003,2026-03-01 09:23:42+00:00,2026-03-01 10:02:45+00:00,39.050000
3,O000004,2026-03-01 09:28:14+00:00,2026-03-01 10:10:41+00:00,42.450000
4,O000005,2026-03-01 11:32:35+00:00,NaT,NaN


after correcting the code, the table now contains 14,494 orders with no repeated IDs or negative delivery durations. there are still 1,101 missing delivery times, which won't be included in the average calculations.

this means 13,393 orders have usable delivery times. the results now match my expected row count, so i'll use this table for the next stage of the analysis.

now i want to calculate the net revenue for each order using payments_clean. i'll treat charges as positive amounts and refunds as negative amounts, then combine them so each order has one net revenue value.

since one order can have multiple payment records, i'll group them by order_id before joining them to the orders table. i'll check that there are no repeated order IDs and compare the total revenue with the original payment records.

**AI Prompt:** i've already removed the 30 duplicate rows in payments_clean. now i want to calculate net revenue for each order using charges minus refunds. can you give me simple python code to group payments_clean by order_id so there's one row per order? also check for repeated order IDs, show the total net revenue and check that the calculation matches the payment records. don't change the original data.

In [15]:
import pandas as pd
import numpy as np

# Get copy of clean payments DataFrame
payments_df = tables["payments_clean"].copy()

# 1. Map charges as positive values and refunds as negative values
payments_df["net_amount"] = np.where(
    payments_df["payment_type"] == "charge",
    payments_df["amount"],
    -payments_df["amount"]
)

# 2. Group by order_id to calculate net revenue per order
order_payments = (
    payments_df.groupby("order_id")["net_amount"]
    .sum()
    .reset_index()
    .rename(columns={"net_amount": "net_revenue"})
)

# Store result in tables dict
tables["order_payments_summary"] = order_payments

# --- Validation Checks ---
# Calculate total net revenue from both sources to check alignment
total_net_revenue_raw = payments_df["net_amount"].sum()
total_net_revenue_grouped = order_payments["net_revenue"].sum()
duplicate_ids = order_payments["order_id"].duplicated().sum()

print("=== Validation Results ===")
print(f"- Total unique orders with payment records: {len(order_payments):,}")
print(f"- Repeated order IDs in grouped table: {duplicate_ids} (Expected: 0)")
print(f"- Sum of raw payments (net): £{total_net_revenue_raw:,.2f}")
print(f"- Sum of grouped order net revenue: £{total_net_revenue_grouped:,.2f}")
print(f"- Calculation matches original payments: {np.isclose(total_net_revenue_raw, total_net_revenue_grouped)}")

# Display preview
display(order_payments.head())

=== Validation Results ===
- Total unique orders with payment records: 14,494
- Repeated order IDs in grouped table: 0 (Expected: 0)
- Sum of raw payments (net): £594,747.49
- Sum of grouped order net revenue: £594,747.49
- Calculation matches original payments: True


,order_id,net_revenue
0,O000001,37.49
1,O000002,31.99
2,O000003,12.50
3,O000004,46.99
4,O000005,12.00


the code produced 14,494 orders with no repeated order IDs. the total net revenue was £594,747.49, which matches the sum of the cleaned payment records.

this suggests that grouping the payments hasn't added or lost any revenue, so i'll use this table when creating the weekly report. however, the calculation assumes that all non-charge payments are refunds and that the amounts are recorded as positive values.

now that i've prepared the restaurant, payment and delivery-time information, i want to combine them into one table to create the weekly operations report.

i'll use orders_with_restaurants as the starting table and add the net revenue and delivery times using order_id. i expect the combined table to have 14,494 rows without any repeated order IDs.

for the final report, i'll group orders by the week they were placed, cuisine and city. i'll count delivered orders, add up their net revenue and calculate the average valid delivery time. i'll use UTC dates and weeks starting on monday.

**AI Prompt:** i've now prepared orders_with_restaurants, order_payments and my corrected delivery time table. can you give me python code to combine them using order_id without adding or losing any orders? i want the final report grouped by week starting monday, cuisine and city, showing the number of delivered orders, their net revenue and average valid delivery time in minutes. use the order_ts date for the week. please check the row counts after joining, repeated order IDs and whether the weekly totals match the order-level data. don't change the original tables.

In [19]:

import pandas as pd
import numpy as np

# 1. Find the corrected delivery-time table
delivery_table = None

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        if {"order_id", "delivery_duration_mins"}.issubset(value.columns):
            if len(value) == 14494 and value["order_id"].is_unique:
                delivery_table = value
                print("Using delivery table:", name)
                break

if delivery_table is None:
    raise ValueError("Corrected delivery-time table not found")

# 2. Combine the cleaned tables
report_data = tables["orders_with_restaurants"].copy()

report_data = report_data.merge(
    order_payments,
    on="order_id",
    how="left",
    validate="one_to_one"
)

report_data = report_data.merge(
    delivery_table[["order_id", "delivery_duration_mins"]],
    on="order_id",
    how="left",
    validate="one_to_one"
)

# 3. Check the combined data
print("Orders after joining:", len(report_data))
print("Repeated order IDs:", report_data["order_id"].duplicated().sum())
print("Missing revenue:", report_data["net_revenue"].isna().sum())

# 4. Create weeks starting on Monday
report_data["order_ts"] = pd.to_datetime(
    report_data["order_ts"], utc=True
)

report_data["week"] = (
    report_data["order_ts"]
    .dt.tz_localize(None)
    .dt.to_period("W-SUN")
    .dt.start_time
)

# 5. Keep orders with usable delivery times
delivered_orders = report_data[
    report_data["delivery_duration_mins"].notna()
].copy()

# 6. Create the weekly report
weekly_report = delivered_orders.groupby(
    ["week", "cuisine", "city"]
).agg(
    number_of_orders=("order_id", "count"),
    net_revenue=("net_revenue", "sum"),
    avg_delivery_minutes=("delivery_duration_mins", "mean")
).reset_index()

# 7. Round the results
weekly_report["net_revenue"] = weekly_report["net_revenue"].round(2)
weekly_report["avg_delivery_minutes"] = weekly_report["avg_delivery_minutes"].round(2)

# 8. Check the final results
print("Total orders:", len(report_data))
print("Orders used:", len(delivered_orders))
print("Weekly orders:", weekly_report["number_of_orders"].sum())
print("Weekly revenue matches:",
      np.isclose(
          weekly_report["net_revenue"].sum(),
          delivered_orders["net_revenue"].sum(),
          atol=0.01 * len(weekly_report)
      ))

# 9. Display the final report
display(weekly_report.head(10))


Using delivery table: order_delivery_perf
Orders after joining: 14494
Repeated order IDs: 0
Missing revenue: 0
Total orders: 14494
Orders used: 13393
Weekly orders: 13393
Weekly revenue matches: True


,week,cuisine,city,number_of_orders,net_revenue,avg_delivery_minutes
0,2026-02-23,Burgers,Birmingham,4,125.99,38.81
1,2026-02-23,Burgers,Bristol,3,99.17,40.20
2,2026-02-23,Burgers,London,9,337.93,36.79
3,2026-02-23,Burgers,London,2,58.49,52.70
4,2026-02-23,Chinese,Birmingham,6,117.46,34.01
5,2026-02-23,Chinese,London,7,284.93,35.07
6,2026-02-23,Chinese,Manchester,1,24.99,33.67
7,2026-02-23,Indian,Birmingham,1,43.99,42.63
8,2026-02-23,Indian,London,7,216.32,30.67
9,2026-02-23,Italian,Birmingham,3,80.98,40.65


the final checks showed that all 14,494 orders were retained after joining, with no repeated order IDs or missing revenue. the weekly report contains 13,393 orders with usable delivery times, and the revenue totals match.

however, i noticed that London appeared twice for the same week and cuisine. this suggests there may still be differences in how some categories are written, so i'll standardise the city and cuisine names before recalculating the report.

**Code source:** ChatGPT generated the correction after i checked the output and identified the repeated categories.

In [20]:

# Standardise city and cuisine names
report_data["city"] = report_data["city"].str.strip().str.title()
report_data["cuisine"] = report_data["cuisine"].str.strip().str.title()

# Recreate the delivered orders table
delivered_orders = report_data[
    report_data["delivery_duration_mins"].notna()
].copy()

# Recalculate the weekly report
weekly_report = delivered_orders.groupby(
    ["week", "cuisine", "city"]
).agg(
    number_of_orders=("order_id", "count"),
    net_revenue=("net_revenue", "sum"),
    avg_delivery_minutes=("delivery_duration_mins", "mean")
).reset_index()

# Round results
weekly_report["net_revenue"] = weekly_report["net_revenue"].round(2)
weekly_report["avg_delivery_minutes"] = weekly_report["avg_delivery_minutes"].round(2)

# Check results
print("Weekly orders:", weekly_report["number_of_orders"].sum())
print("Repeated week/cuisine/city groups:",
      weekly_report.duplicated(["week", "cuisine", "city"]).sum())

display(weekly_report.head(10))


Weekly orders: 13393
Repeated week/cuisine/city groups: 0


,week,cuisine,city,number_of_orders,net_revenue,avg_delivery_minutes
0,2026-02-23,Burgers,Birmingham,4,125.99,38.81
1,2026-02-23,Burgers,Bristol,3,99.17,40.20
2,2026-02-23,Burgers,London,11,396.42,39.68
3,2026-02-23,Chinese,Birmingham,6,117.46,34.01
4,2026-02-23,Chinese,London,7,284.93,35.07
5,2026-02-23,Chinese,Manchester,1,24.99,33.67
6,2026-02-23,Indian,Birmingham,1,43.99,42.63
7,2026-02-23,Indian,London,7,216.32,30.67
8,2026-02-23,Italian,Birmingham,7,138.94,36.24
9,2026-02-23,Italian,Leeds,1,49.49,32.02


after standardising the city and cuisine names, the weekly report still contains 13,393 orders and there are no repeated week, cuisine and city combinations.

the London records that previously appeared separately have now been combined. for example, Burgers in London now has 11 orders for the week beginning 23 February, with £396.42 in net revenue and an average delivery time of 39.68 minutes.

this suggests the grouping is now consistent and the final table is ready for reporting.

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Add prompts here.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.